# Scaffold Training Three Seeds

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 58.


In [ ]:
# ============================================================
# Corrected nonchiral scaffold — Linear MLP, Seed 42
# Train-only preprocessing already saved.
# Checkpoint after every completed epoch; automatic resume.
# ============================================================

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json
import copy
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display, FileLink

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)
FEATURES = PROTOCOL / "train_only_ipc_log1p_20260921_190520_291433"
OUT = PROTOCOL / "controlled_rerun_v1" / "linear_mlp" / "seed_42"

SEED = 42
CONFIG = {
    "model": "linear_mlp",
    "seed": SEED,
    "input_dim": 2250,
    "encoder": [384, 384, 192],
    "pair": "z1,z2,abs(z1-z2),z1*z2",
    "dropout": 0.35,
    "batch_size": 512,
    "epochs": 120,
    "patience": 18,
    "lr": 0.0007,
    "weight_decay": 0.0001,
    "clip": 5.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 5,
    "selection": "cal_macro83",
    "class_weights": "sqrt(max_count/(count+1)); clip[0.5,3]; divide_by_mean",
    "training_encoder_scope": "all Train drugs, recomputed per pair minibatch",
    "initialization": "fresh; no Cold-start weights",
}

assert torch.cuda.is_available(), "Enable a GPU before training."
device = torch.device("cuda")

required = [
    FEATURES / "molecular_features.npz",
    *[FEATURES / f"{name}_pairs.csv"
      for name in ("train", "calibration", "test")],
]
for path in required:
    assert path.is_file(), f"Missing: {path}"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(obj, path):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    os.replace(temporary, path)

def save_checkpoint(obj, path):
    temporary = path.with_name(path.name + ".tmp")
    torch.save(obj, temporary)
    os.replace(temporary, path)

def cpu_state(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }

OUT.mkdir(parents=True, exist_ok=True)
identity = {
    "config": CONFIG,
    "file_sha256": {path.name: sha256_file(path) for path in required},
    "torch": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0),
}
identity_path = OUT / "run_identity.json"
if identity_path.exists():
    assert json.loads(identity_path.read_text()) == identity, (
        "Saved run identity differs. Stop rather than mix experiments."
    )
else:
    save_json(identity, identity_path)

with np.load(required[0], allow_pickle=False) as saved:
    x_np = saved["x"].copy()
    all_drugs = saved["all_drugs"].astype(str)
    fit_rows = saved["train_drug_indices"].astype(np.int64)

assert x_np.shape == (1659, 2250)
assert np.isfinite(x_np).all()

frames = {
    name: pd.read_csv(
        FEATURES / f"{name}_pairs.csv", keep_default_na=False
    )
    for name in ("train", "calibration", "test")
}

expected = {
    "train": (91477, 84),
    "calibration": (16151, 83),
    "test": (11142, 56),
}

pair_arrays, labels = {}, {}
for name, frame in frames.items():
    assert (len(frame), frame.label_id.nunique()) == expected[name]
    pair_arrays[name] = frame[
        ["drug1_id", "drug2_id"]
    ].to_numpy(dtype=np.int64)
    labels[name] = frame.label_id.to_numpy(dtype=np.int64)
    assert pair_arrays[name].min() >= 0
    assert pair_arrays[name].max() < len(all_drugs)
    assert labels[name].min() >= 0 and labels[name].max() < 84

    for column, index_column in (
        ("Drug1", "drug1_id"), ("Drug2", "drug2_id")
    ):
        assert np.array_equal(
            frame[column].astype(str).to_numpy(),
            all_drugs[frame[index_column].to_numpy(dtype=np.int64)],
        ), f"{name}: drug-feature alignment failed."

train_ids = np.unique(pair_arrays["train"])
np.testing.assert_array_equal(np.sort(fit_rows), train_ids)
assert len(train_ids) == 1276

dev_ids = set(train_ids) | set(pair_arrays["calibration"].ravel())
assert not dev_ids.intersection(pair_arrays["test"].ravel())

# Map training pair indices into the Train-only encoder tensor.
global_to_train = np.full(len(all_drugs), -1, dtype=np.int64)
global_to_train[train_ids] = np.arange(len(train_ids))
train_local = global_to_train[pair_arrays["train"]]
assert (train_local >= 0).all()

x_all = torch.as_tensor(x_np, dtype=torch.float32, device=device)
x_train = x_all[torch.as_tensor(train_ids, device=device)]
train_pairs = torch.as_tensor(train_local, device=device)
y_train = torch.as_tensor(labels["train"], device=device)

counts = np.bincount(labels["train"], minlength=84)
assert (counts > 0).all()
weights = np.sqrt(counts.max() / (counts + 1))
weights = np.clip(weights, 0.5, 3.0)
weights /= weights.mean()

pd.DataFrame({
    "label_id": np.arange(84),
    "train_count": counts,
    "class_weight": weights,
}).to_csv(OUT / "train_class_weights.csv", index=False)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(2250, 384),
            nn.Linear(384, 384),
            nn.Linear(384, 192),
        ])
        self.norms = nn.ModuleList([
            nn.BatchNorm1d(384),
            nn.BatchNorm1d(384),
            nn.BatchNorm1d(192),
        ])
        self.dropout = nn.Dropout(0.35)

    def forward(self, x):
        for i, (layer, norm) in enumerate(zip(self.layers, self.norms)):
            x = norm(layer(x))
            if i < 2:
                x = self.dropout(torch.relu(x))
        return x

class LinearMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 84),
        )

    def classify(self, z, pairs):
        a, b = z[pairs[:, 0]], z[pairs[:, 1]]
        return self.classifier(
            torch.cat([a, b, (a - b).abs(), a * b], dim=1)
        )

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

model = LinearMLP().to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max",
    factor=CONFIG["scheduler_factor"],
    patience=CONFIG["scheduler_patience"],
)
criterion = nn.CrossEntropyLoss(
    weight=torch.as_tensor(weights, dtype=torch.float32, device=device)
)

@torch.no_grad()
def evaluate(name):
    model.eval()
    z = model.encoder(x_all)
    assert torch.isfinite(z).all()
    predictions = []

    for start in range(0, len(labels[name]), 2048):
        pair_batch = torch.as_tensor(
            pair_arrays[name][start:start + 2048], device=device
        )
        logits = model.classify(z, pair_batch)
        assert torch.isfinite(logits).all()
        predictions.append(logits.argmax(1).cpu().numpy())

    predictions = np.concatenate(predictions)
    observed = np.unique(labels[name])
    return {
        "accuracy": float(accuracy_score(labels[name], predictions)),
        "macro_observed": float(f1_score(
            labels[name], predictions, labels=observed,
            average="macro", zero_division=0,
        )),
        "macro84": float(f1_score(
            labels[name], predictions, labels=np.arange(84),
            average="macro", zero_division=0,
        )),
        "weighted_f1": float(f1_score(
            labels[name], predictions, average="weighted",
            zero_division=0,
        )),
    }, predictions

last_path = OUT / "last_checkpoint.pt"
best_path = OUT / "best_checkpoint.pt"
result_path = OUT / "final_result.json"

start_epoch, best_epoch, wait = 1, 0, 0
best_score = -1.0
best_model = None
history = []

if not result_path.exists():
    if last_path.exists():
        checkpoint = torch.load(
            last_path, map_location="cpu", weights_only=True
        )
        assert checkpoint["identity"] == identity
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = checkpoint["epoch"] + 1
        best_epoch = checkpoint["best_epoch"]
        best_score = checkpoint["best_score"]
        best_model = checkpoint["best_model"]
        wait = checkpoint["wait"]
        history = checkpoint["history"]
        torch.set_rng_state(checkpoint["torch_rng"])
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
        print("Resuming after epoch:", start_epoch - 1, flush=True)

    print("Device:", torch.cuda.get_device_name(0), flush=True)
    print("Parameters:", sum(p.numel() for p in model.parameters()), flush=True)
    print("Train-only encoder drugs:", len(train_ids), flush=True)
    print("Selection: CAL Macro83 | Seed: 42", flush=True)

    try:
        for epoch in range(start_epoch, CONFIG["epochs"] + 1):
            if wait >= CONFIG["patience"]:
                print("Early stopping reached.", flush=True)
                break

            model.train()
            order = torch.randperm(len(y_train)).to(device)
            loss_total = 0.0

            for batch_number, start in enumerate(
                range(0, len(y_train), CONFIG["batch_size"]), start=1
            ):
                indices = order[start:start + CONFIG["batch_size"]]
                optimizer.zero_grad(set_to_none=True)

                # BatchNorm sees Train drugs only while training.
                z = model.encoder(x_train)
                logits = model.classify(z, train_pairs[indices])
                loss = criterion(logits, y_train[indices])
                assert torch.isfinite(loss), "Non-finite training loss."

                loss.backward()
                nn.utils.clip_grad_norm_(
                    model.parameters(), CONFIG["clip"],
                    error_if_nonfinite=True,
                )
                optimizer.step()
                loss_total += loss.item() * len(indices)

                if batch_number == 1 or batch_number % 60 == 0:
                    print(
                        f"Epoch {epoch:03d} | "
                        f"Train {min(start + len(indices), len(y_train)):,}"
                        f"/{len(y_train):,} | Loss {loss.item():.5f}",
                        flush=True,
                    )

            cal, _ = evaluate("calibration")
            score = cal["macro_observed"]

            if score > best_score:
                best_score, best_epoch, wait = score, epoch, 0
                best_model = cpu_state(model)
            else:
                wait += 1

            scheduler.step(score)
            history.append({
                "epoch": epoch,
                "train_loss": loss_total / len(y_train),
                "cal_macro83": score,
                "cal_macro84": cal["macro84"],
                "cal_accuracy": cal["accuracy"],
                "lr": optimizer.param_groups[0]["lr"],
            })

            # Includes best weights so resume does not depend on a
            # separately written best-checkpoint file.
            save_checkpoint({
                "identity": identity,
                "epoch": epoch,
                "model": cpu_state(model),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_model": best_model,
                "best_score": best_score,
                "best_epoch": best_epoch,
                "wait": wait,
                "history": history,
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all(),
            }, last_path)

            pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
            print(
                f" Epoch {epoch} saved | CAL Macro83={score:.6f}"
                f" | Best={best_score:.6f} | Wait={wait}/18",
                flush=True,
            )

    except KeyboardInterrupt:
        print(
            "\nStopped. Rerun this cell to resume from the last "
            "completed epoch. The interrupted epoch will be repeated.",
            flush=True,
        )
        raise

    assert best_model is not None
    save_checkpoint({
        "identity": identity,
        "model": best_model,
        "best_epoch": best_epoch,
        "cal_macro83": best_score,
    }, best_path)

    restored = torch.load(best_path, map_location="cpu", weights_only=True)
    model.load_state_dict(restored["model"])

    cal_check, _ = evaluate("calibration")
    gap = abs(cal_check["macro_observed"] - best_score)
    assert gap <= 1e-12, f"Calibration checkpoint mismatch: {gap}"

    print("Evaluating corrected Cold Scaffold Test...", flush=True)
    test, predictions = evaluate("test")

    prediction_frame = frames["test"].copy()
    prediction_frame["pred_label_id"] = predictions
    prediction_frame["correct"] = predictions == labels["test"]
    prediction_path = OUT / "test_predictions.csv"
    prediction_frame.to_csv(prediction_path, index=False)

    checked = pd.read_csv(prediction_path)
    np.testing.assert_array_equal(checked.pred_label_id, predictions)

    result = {
        "model": "linear_mlp",
        "seed": SEED,
        "protocol": "both_nonchiral_scaffolds_unseen",
        "completed_epochs": history[-1]["epoch"],
        "best_epoch": best_epoch,
        "cal_macro83": best_score,
        "checkpoint_gap": gap,
        "test_rows": len(predictions),
        "test_observed_classes": len(np.unique(labels["test"])),
        "test_accuracy": test["accuracy"],
        "test_macro56": test["macro_observed"],
        "test_macro84": test["macro84"],
        "test_weighted_f1": test["weighted_f1"],
        "best_checkpoint_sha256": sha256_file(best_path),
        "test_predictions_sha256": sha256_file(prediction_path),
    }
    pd.DataFrame([result]).to_csv(OUT / "final_result.csv", index=False)
    save_json(result, result_path)

else:
    result = json.loads(result_path.read_text())
    assert sha256_file(best_path) == result["best_checkpoint_sha256"]
    assert sha256_file(OUT / "test_predictions.csv") == (
        result["test_predictions_sha256"]
    )
    print("Completed result already exists; training skipped.", flush=True)

archive_path = OUT.parent / "SCAFFOLD_LINEAR_MLP_SEED_42_COMPLETE.zip"
temporary_zip = archive_path.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temporary_zip, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and not path.name.endswith(".tmp"):
            archive.write(path, arcname=str(path.relative_to(OUT.parent)))

with zipfile.ZipFile(temporary_zip) as archive:
    assert archive.testzip() is None

os.replace(temporary_zip, archive_path)

print("\n LINEAR MLP — SEED 42 COMPLETED")
print(json.dumps(result, indent=2))
print(" ZIP integrity passed.")
print(" No Graph run or Seed 43 started.")
print("Download:", archive_path)
display(FileLink(str(archive_path.relative_to("/kaggle/working"))))

Source cell: 60.


In [ ]:
# ============================================================
# CORRECTED SCAFFOLD — GRAPH, SEED 42
# Robustly recover the REAL executed MLP trainer
# Replace ONLY the Encoder with GCN
# Keep same splits / classifier / loss / trainer
# Train interaction edges only
# ============================================================

import ast
import subprocess
import sys
import importlib.util
from pathlib import Path


# ============================================================
# 1) RECOVER THE REAL EXECUTED MLP TRAINING CELL
# ============================================================

shell = get_ipython()

assert shell is not None, (
    "Run this cell inside the same Kaggle notebook/session "
    "where the MLP Seed 42 cell was executed."
)

history_cells = list(
    shell.history_manager.input_hist_raw
)

candidates = []


for hist_idx, source in enumerate(history_cells):

    if not isinstance(source, str):
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    # --------------------------------------------------------
    # IMPORTANT:
    # Detect REAL class definitions.
    # Strings containing "class Encoder(...)" do NOT count.
    # --------------------------------------------------------
    top_level_classes = {
        node.name: node
        for node in tree.body
        if isinstance(node, ast.ClassDef)
    }

    if "Encoder" not in top_level_classes:
        continue

    if "LinearMLP" not in top_level_classes:
        continue

    # Additional fingerprints of the REAL MLP trainer
    required_terms = [
        'CONFIG["patience"]',
        "SCAFFOLD_LINEAR_MLP_SEED_42_COMPLETE.zip",
        "last_checkpoint.pt",
        "test_predictions.csv",
    ]

    if not all(
        term in source
        for term in required_terms
    ):
        continue

    candidates.append(
        {
            "history_index": hist_idx,
            "source": source,
            "tree": tree,
            "classes": top_level_classes,
        }
    )


assert candidates, (
    "Original MLP training cell not found in current session history.\n"
    "Do not retrain the completed MLP.\n"
    "The MLP seed-42 source must be available in the same notebook session."
)


# Latest REAL MLP trainer
selected = candidates[-1]

original_source = selected["source"]
original_tree = selected["tree"]
original_classes = selected["classes"]


print(
    " REAL MLP trainer recovered from history index:",
    selected["history_index"],
    flush=True,
)

print(
    "Source length:",
    len(original_source),
    flush=True,
)

print(
    "Top-level classes:",
    list(original_classes.keys()),
    flush=True,
)


assert "Encoder" in original_classes
assert "LinearMLP" in original_classes
assert "ControlledGCN" not in original_classes


# ============================================================
# 2) CHECK / INSTALL TORCH GEOMETRIC
# ============================================================

if importlib.util.find_spec("torch_geometric") is None:

    print(
        "Installing torch-geometric...",
        flush=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "torch-geometric==2.8.0.post1",
        ],
        check=True,
        timeout=300,
    )


print(
    " torch_geometric available.",
    flush=True,
)


# ============================================================
# 3) GRAPH ENCODER SOURCE
# ============================================================

graph_encoder = r'''
from torch_geometric.nn import GCNConv


# ============================================================
# GRAPH EDGES
# ============================================================
#
# Only TRAIN interaction rows are used as graph edges.
#
# Forward:
#   Drug1 -> Drug2
#
# Reverse:
#   Drug2 -> Drug1
#
# Duplicate edges are deliberately retained to preserve the
# exact training-row multiplicity.
# ============================================================

train_pairs_global_cpu = torch.as_tensor(
    pair_arrays["train"],
    dtype=torch.long,
)

train_pairs_local_cpu = torch.as_tensor(
    train_local,
    dtype=torch.long,
)


# ------------------------------------------------------------
# Global graph indexing
# Used at evaluation time with full feature cache
# ------------------------------------------------------------

edge_global = torch.cat(
    [
        train_pairs_global_cpu.T,
        train_pairs_global_cpu.flip(1).T,
    ],
    dim=1,
).contiguous().to(device)


# ------------------------------------------------------------
# Local TRAIN-drug indexing
# Used during training
# ------------------------------------------------------------

edge_train = torch.cat(
    [
        train_pairs_local_cpu.T,
        train_pairs_local_cpu.flip(1).T,
    ],
    dim=1,
).contiguous().to(device)


# ============================================================
# GRAPH QA
# ============================================================

assert edge_global.shape == (
    2,
    2 * len(labels["train"]),
), (
    edge_global.shape,
    len(labels["train"]),
)

assert edge_train.shape == edge_global.shape

assert edge_train.min().item() >= 0

assert edge_train.max().item() < len(train_ids)


# ------------------------------------------------------------
# Verify local indexing resolves exactly to global indexing
# ------------------------------------------------------------

train_ids_tensor = torch.as_tensor(
    train_ids,
    dtype=torch.long,
    device=device,
)

assert torch.equal(
    train_ids_tensor[edge_train],
    edge_global,
), (
    "Local Train graph and Global Train graph do not match."
)


# ------------------------------------------------------------
# Verify graph contains Train drugs only
# ------------------------------------------------------------

edge_drugs = set(
    edge_global
    .detach()
    .cpu()
    .numpy()
    .ravel()
    .tolist()
)

assert edge_drugs == set(
    train_ids.tolist()
), (
    "Train graph drug set does not equal Train drug IDs."
)


# ------------------------------------------------------------
# Critical leakage check:
# No TEST drug may be incident to a TRAIN interaction edge
# ------------------------------------------------------------

test_drugs = set(
    pair_arrays["test"]
    .ravel()
    .tolist()
)

assert not edge_drugs.intersection(
    test_drugs
), (
    "Leakage detected: TEST drug found in TRAIN graph edges."
)


print(
    "Directed Train edges:",
    edge_global.shape[1],
    flush=True,
)

print(
    "Train graph drugs:",
    len(edge_drugs),
    flush=True,
)

print(
    "Test drugs incident to Train edges: 0",
    flush=True,
)


# ============================================================
# CONTROLLED GCN ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.layers = nn.ModuleList(
            [
                GCNConv(
                    2250,
                    384,
                    cached=False,
                    add_self_loops=True,
                    normalize=True,
                ),

                GCNConv(
                    384,
                    384,
                    cached=False,
                    add_self_loops=True,
                    normalize=True,
                ),

                GCNConv(
                    384,
                    192,
                    cached=False,
                    add_self_loops=True,
                    normalize=True,
                ),
            ]
        )

        self.norms = nn.ModuleList(
            [
                nn.BatchNorm1d(384),
                nn.BatchNorm1d(384),
                nn.BatchNorm1d(192),
            ]
        )

        self.dropout = nn.Dropout(
            0.35
        )


    def forward(self, x):

        # ----------------------------------------------------
        # TRAIN MODE
        # ----------------------------------------------------
        if self.training:

            # BatchNorm and message passing see TRAIN drugs only
            assert x.shape[0] == len(train_ids), (
                x.shape,
                len(train_ids),
            )

            edges = edge_train


        # ----------------------------------------------------
        # EVALUATION MODE
        # ----------------------------------------------------
        else:

            # Full feature cache may contain held-out drugs,
            # BUT interaction edges remain TRAIN-only.
            #
            # Held-out drugs therefore have no training
            # interaction neighbors; GCNConv only introduces
            # their self-loop.
            assert x.shape[0] == len(all_drugs), (
                x.shape,
                len(all_drugs),
            )

            edges = edge_global


        # ----------------------------------------------------
        # GCN STACK
        # ----------------------------------------------------

        for i, (layer, norm) in enumerate(
            zip(
                self.layers,
                self.norms,
            )
        ):

            x = layer(
                x,
                edges,
            )

            x = norm(
                x
            )

            if i < 2:

                x = torch.relu(
                    x
                )

                x = self.dropout(
                    x
                )


        return x
'''


# ============================================================
# 4) REPLACE ONLY THE REAL Encoder CLASS
# ============================================================

encoder_node = original_classes[
    "Encoder"
]


# Work by complete line ranges rather than fragile string index.
source_lines = original_source.splitlines(
    keepends=True
)


encoder_start_line = (
    encoder_node.lineno - 1
)

encoder_end_line = (
    encoder_node.end_lineno
)


old_encoder_text = "".join(
    source_lines[
        encoder_start_line:
        encoder_end_line
    ]
)


assert old_encoder_text.lstrip().startswith(
    "class Encoder"
), (
    "Failed to locate the real Encoder class."
)


graph_source = (
    "".join(
        source_lines[
            :encoder_start_line
        ]
    )
    +
    graph_encoder.strip()
    +
    "\n\n"
    +
    "".join(
        source_lines[
            encoder_end_line:
        ]
    )
)


# ============================================================
# 5) QA BEFORE RENAMING MODEL CLASS
# ============================================================

tree_check = ast.parse(
    graph_source
)


classes_before_rename = {
    node.name
    for node in tree_check.body
    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert "Encoder" in classes_before_rename

assert "LinearMLP" in classes_before_rename

assert "ControlledGCN" not in classes_before_rename


print(
    " Only the real Encoder class was replaced.",
    flush=True,
)


# ============================================================
# 6) RENAME MODEL / OUTPUTS
# ============================================================

# Change model class
graph_source = graph_source.replace(
    "LinearMLP",
    "ControlledGCN",
)


# Change experiment subdirectory / model identity
graph_source = graph_source.replace(
    "linear_mlp",
    "graph",
)


# Human-readable printed model name
graph_source = graph_source.replace(
    "LINEAR MLP",
    "GRAPH",
)


# Separate ZIP output
graph_source = graph_source.replace(
    "SCAFFOLD_LINEAR_MLP_SEED_42_COMPLETE.zip",
    "SCAFFOLD_GRAPH_SEED_42_COMPLETE.zip",
)


# Final status message
graph_source = graph_source.replace(
    " No Graph run or Seed 43 started.",
    " Graph Seed 42 finished. No Seed 43 started.",
)


# ============================================================
# 7) ADD GRAPH-SPECIFIC CONFIG
# ============================================================

anchor = "assert torch.cuda.is_available()"


assert graph_source.count(
    anchor
) == 1, (
    "Expected exactly one CUDA assertion anchor, "
    f"found {graph_source.count(anchor)}."
)


graph_config = '''
# ============================================================
# GRAPH-SPECIFIC CONTROLLED-RERUN CONFIG
# ============================================================

CONFIG.update({

    "model":
        "graph",

    "graph_edges":
        "Train rows plus reverse; duplicates retained",

    "graph_layer":
        "GCNConv",

    "gcn_normalize":
        True,

    "gcn_add_self_loops":
        True,

    "gcn_cached":
        False,

    "evaluation_edges":
        "Train interaction edges only",

    "heldout_drug_graph_access":
        "self-loop only; no held-out interaction edges",

    "training_encoder_scope":
        "all Train drugs, recomputed per pair minibatch",
})


import torch_geometric

CONFIG[
    "torch_geometric_version"
] = str(
    torch_geometric.__version__
)

'''


graph_source = graph_source.replace(
    anchor,
    graph_config
    +
    "\n"
    +
    anchor,
    1,
)


# ============================================================
# 8) ADD PARAMETER COUNT LOGGING
# ============================================================
#
# IMPORTANT:
# Do NOT hard-code a guessed parameter count.
# Record actual architecture count for reproducibility.
# ============================================================

model_anchor = (
    "model = ControlledGCN().to(device)"
)


assert graph_source.count(
    model_anchor
) == 1, (
    "Could not uniquely locate model construction. "
    f"Found {graph_source.count(model_anchor)} occurrences."
)


parameter_logging = r'''

# ============================================================
# PARAMETER COUNT — REPRODUCIBILITY RECORD
# ============================================================

actual_parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "GRAPH total parameter count:",
    actual_parameter_count,
    flush=True,
)

print(
    "GRAPH trainable parameter count:",
    trainable_parameter_count,
    flush=True,
)

CONFIG[
    "total_parameter_count"
] = int(
    actual_parameter_count
)

CONFIG[
    "trainable_parameter_count"
] = int(
    trainable_parameter_count
)

'''


graph_source = graph_source.replace(
    model_anchor,
    model_anchor
    +
    parameter_logging,
    1,
)


# ============================================================
# 9) FINAL GENERATED-SCRIPT QA
# ============================================================

ast.parse(
    graph_source
)


# ------------------------------------------------------------
# Verify class definitions
# ------------------------------------------------------------

final_tree = ast.parse(
    graph_source
)


final_classes = {
    node.name
    for node in final_tree.body
    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert "Encoder" in final_classes

assert "ControlledGCN" in final_classes

assert "LinearMLP" not in final_classes


# ------------------------------------------------------------
# Verify experiment identity
# ------------------------------------------------------------

assert '"model": "graph"' in graph_source

assert '/ "graph" / "seed_42"' in graph_source

assert (
    "SCAFFOLD_GRAPH_SEED_42_COMPLETE.zip"
    in graph_source
)


# ------------------------------------------------------------
# Verify original trainer settings remain present
# ------------------------------------------------------------

assert "cal_macro83" in graph_source

assert '"epochs": 120' in graph_source

assert '"patience": 18' in graph_source

assert "last_checkpoint.pt" in graph_source

assert "test_predictions.csv" in graph_source


# ------------------------------------------------------------
# Verify graph protocol
# ------------------------------------------------------------

assert (
    "Test drugs incident to Train edges: 0"
    in graph_source
)

assert (
    '"evaluation_edges":'
    in graph_source
)

assert (
    "Train interaction edges only"
    in graph_source
)


print(
    " Generated Graph script passes syntax QA.",
    flush=True,
)

print(
    " Controlled trainer settings preserved.",
    flush=True,
)

print(
    " Output namespace changed to graph/seed_42.",
    flush=True,
)


# ============================================================
# 10) LOCATE CORRECTED SCAFFOLD PROTOCOL
# ============================================================

protocol = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)


assert protocol.is_dir(), (
    f"Missing corrected scaffold protocol:\n{protocol}"
)


graph_out = (
    protocol
    /
    "controlled_rerun_v1"
    /
    "graph"
    /
    "seed_42"
)


graph_out.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Graph output directory:",
    graph_out,
    flush=True,
)


# ============================================================
# 11) SAVE GENERATED GRAPH TRAINER
# ============================================================

script_path = (
    graph_out
    /
    "train_graph_seed42.py"
)


if script_path.exists():

    existing_script = script_path.read_text(
        encoding="utf-8"
    )

    if existing_script != graph_source:

        # Preserve previous script rather than overwrite silently.
        backup_path = (
            graph_out
            /
            "train_graph_seed42_PREVIOUS.py"
        )

        if not backup_path.exists():

            backup_path.write_text(
                existing_script,
                encoding="utf-8",
            )

            print(
                " Previous Graph script preserved as:",
                backup_path,
                flush=True,
            )

        # Write corrected script explicitly
        script_path.write_text(
            graph_source,
            encoding="utf-8",
        )

        print(
            " Corrected Graph script replaced old generated wrapper.",
            flush=True,
        )

    else:

        print(
            " Existing Graph script is identical.",
            flush=True,
        )

else:

    script_path.write_text(
        graph_source,
        encoding="utf-8",
    )

    print(
        " Graph script saved:",
        script_path,
        flush=True,
    )


# ============================================================
# 12) PRESERVE EXACT RECOVERED MLP TRAINER
# ============================================================

reference_path = (
    graph_out
    /
    "reference_mlp_training_cell.txt"
)


if reference_path.exists():

    existing_reference = reference_path.read_text(
        encoding="utf-8"
    )

    assert existing_reference == original_source, (
        "A different MLP reference trainer already exists.\n"
        "Stopping to avoid mixing experimental protocols."
    )

else:

    reference_path.write_text(
        original_source,
        encoding="utf-8",
    )


print(
    " Original MLP trainer preserved:",
    reference_path,
    flush=True,
)


# ============================================================
# 13) PRE-RUN SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80,
    flush=True,
)

print(
    "CORRECTED SCAFFOLD GRAPH — SEED 42",
    flush=True,
)

print(
    "=" * 80,
    flush=True,
)

print(
    " Exact executed MLP trainer recovered",
    flush=True,
)

print(
    " Same corrected scaffold splits",
    flush=True,
)

print(
    " Same molecular features",
    flush=True,
)

print(
    " Same classifier",
    flush=True,
)

print(
    " Same loss / optimizer / trainer",
    flush=True,
)

print(
    " Only Encoder replaced with GCN",
    flush=True,
)

print(
    " Train interaction edges only",
    flush=True,
)

print(
    " Reverse Train edges added",
    flush=True,
)

print(
    " Duplicate Train edges retained",
    flush=True,
)

print(
    " No TEST interaction edge enters graph",
    flush=True,
)

print(
    " Separate graph/seed_42 outputs",
    flush=True,
)

print(
    "\nStarting GRAPH — Seed 42...",
    flush=True,
)


# ============================================================
# 14) EXECUTE IN SEPARATE NAMESPACE
# ============================================================

graph_namespace = {
    "__name__":
        "__main__",

    "__file__":
        str(
            script_path
        ),
}


exec(
    compile(
        graph_source,
        str(script_path),
        "exec",
    ),
    graph_namespace,
)

Source cell: 61.


In [ ]:
# ============================================================
# CORRECTED NONCHIRAL SCAFFOLD — LINEAR MLP, SEED 43
#
# Reuse the exact Seed-42 MLP trainer saved during Graph-42.
# Change ONLY:
#   - random seed 42 -> 43
#   - output folder seed_42 -> seed_43
#   - ZIP/output labels
#
# Same:
#   - corrected scaffold split
#   - train-only preprocessing
#   - features
#   - architecture
#   - pair representation
#   - class weights
#   - optimizer
#   - scheduler
#   - max epochs = 120
#   - patience = 18
# ============================================================

import ast
from pathlib import Path


# ============================================================
# 1) PATHS
# ============================================================

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)

REFERENCE_42 = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "graph"
    / "seed_42"
    / "reference_mlp_training_cell.txt"
)

assert REFERENCE_42.is_file(), (
    "Missing exact Seed-42 MLP reference trainer:\n"
    f"{REFERENCE_42}\n\n"
    "Do not reconstruct manually."
)


# ============================================================
# 2) LOAD EXACT EXECUTED SEED-42 TRAINER
# ============================================================

source42 = REFERENCE_42.read_text(
    encoding="utf-8"
)

ast.parse(source42)


# ============================================================
# 3) VERIFY REFERENCE IDENTITY
# ============================================================

assert "class Encoder(nn.Module):" in source42
assert "class LinearMLP(nn.Module):" in source42

assert "SEED = 42" in source42

assert '"epochs": 120' in source42
assert '"patience": 18' in source42

assert (
    '"class_weights": '
    '"sqrt(max_count/(count+1)); clip[0.5,3]; divide_by_mean"'
    in source42
)

assert (
    '"pair": "z1,z2,abs(z1-z2),z1*z2"'
    in source42
)

assert (
    "SCAFFOLD_LINEAR_MLP_SEED_42_COMPLETE.zip"
    in source42
)


print(
    " Exact Seed-42 MLP trainer loaded.",
    flush=True,
)


# ============================================================
# 4) CREATE SEED-43 SOURCE
# ============================================================

source43 = source42


# ------------------------------------------------------------
# Seed
# ------------------------------------------------------------

assert source43.count(
    "SEED = 42"
) == 1

source43 = source43.replace(
    "SEED = 42",
    "SEED = 43",
    1,
)


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

source43 = source43.replace(
    '"seed_42"',
    '"seed_43"',
)


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

source43 = source43.replace(
    "SCAFFOLD_LINEAR_MLP_SEED_42_COMPLETE.zip",
    "SCAFFOLD_LINEAR_MLP_SEED_43_COMPLETE.zip",
)


# ------------------------------------------------------------
# Human-readable messages only
# ------------------------------------------------------------

source43 = source43.replace(
    "Selection: CAL Macro83 | Seed: 42",
    "Selection: CAL Macro83 | Seed: 43",
)

source43 = source43.replace(
    " LINEAR MLP — SEED 42 COMPLETED",
    " LINEAR MLP — SEED 43 COMPLETED",
)

source43 = source43.replace(
    " No Graph run or Seed 43 started.",
    " No Graph run or Seed 44 started.",
)


# ============================================================
# 5) VERIFY SEED-43 SCRIPT
# ============================================================

ast.parse(source43)


assert "SEED = 43" in source43
assert "SEED = 42" not in source43

assert '"seed_43"' in source43

assert (
    "SCAFFOLD_LINEAR_MLP_SEED_43_COMPLETE.zip"
    in source43
)

assert (
    "SCAFFOLD_LINEAR_MLP_SEED_42_COMPLETE.zip"
    not in source43
)

assert "class LinearMLP(nn.Module):" in source43

assert '"epochs": 120' in source43
assert '"patience": 18' in source43

assert (
    '"pair": "z1,z2,abs(z1-z2),z1*z2"'
    in source43
)

assert (
    '"class_weights": '
    '"sqrt(max_count/(count+1)); clip[0.5,3]; divide_by_mean"'
    in source43
)


print(
    " Seed-43 trainer generated.",
    flush=True,
)

print(
    " Same architecture / split / loss / hyperparameters.",
    flush=True,
)


# ============================================================
# 6) SAVE SEED-43 TRAINER
# ============================================================

OUT43 = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "linear_mlp"
    / "seed_43"
)

OUT43.mkdir(
    parents=True,
    exist_ok=True,
)

SCRIPT43 = (
    OUT43
    / "train_linear_mlp_seed43.py"
)


if SCRIPT43.exists():

    existing = SCRIPT43.read_text(
        encoding="utf-8"
    )

    assert existing == source43, (
        "A DIFFERENT Seed-43 MLP trainer already exists.\n"
        "Stopping rather than mixing experiments."
    )

    print(
        " Existing Seed-43 trainer matches exactly.",
        flush=True,
    )

else:

    SCRIPT43.write_text(
        source43,
        encoding="utf-8",
    )

    print(
        " Seed-43 trainer saved:",
        SCRIPT43,
        flush=True,
    )


# Also preserve exact Seed-42 reference used to generate it
REFERENCE_COPY = (
    OUT43
    / "reference_seed42_mlp_training_cell.txt"
)

if REFERENCE_COPY.exists():

    assert (
        REFERENCE_COPY.read_text(
            encoding="utf-8"
        )
        ==
        source42
    )

else:

    REFERENCE_COPY.write_text(
        source42,
        encoding="utf-8",
    )


# ============================================================
# 7) START MLP SEED 43
# ============================================================

print(
    "\n"
    + "=" * 80,
    flush=True,
)

print(
    "CORRECTED SCAFFOLD — LINEAR MLP — SEED 43",
    flush=True,
)

print(
    "=" * 80,
    flush=True,
)

print(
    "Max epochs: 120",
    flush=True,
)

print(
    "Early stopping patience: 18",
    flush=True,
)

print(
    " Fresh Seed 43 initialization",
    flush=True,
)

print(
    " Train-only preprocessing unchanged",
    flush=True,
)

print(
    " Corrected nonchiral scaffold split unchanged",
    flush=True,
)

print(
    " Class weighting unchanged",
    flush=True,
)

print(
    "\nStarting LINEAR MLP — Seed 43...",
    flush=True,
)


namespace43 = {
    "__name__": "__main__",
    "__file__": str(SCRIPT43),
}


exec(
    compile(
        source43,
        str(SCRIPT43),
        "exec",
    ),
    namespace43,
)

Source cell: 62.


In [ ]:
# ============================================================
# CORRECTED NONCHIRAL SCAFFOLD — GRAPH, SEED 43
#
# Source of truth:
# exact generated MLP Seed-43 trainer.
#
# Replace ONLY Encoder:
#   Linear encoder -> GCN encoder
#
# Preserve:
#   corrected scaffold splits
#   train-only features
#   pair classifier
#   loss
#   class weights
#   optimizer
#   scheduler
#   epochs = 120
#   patience = 18
#
# GRAPH:
#   TRAIN interaction edges ONLY
#   + reversed TRAIN edges
#   duplicates retained
#   no TEST interaction edges
# ============================================================

import ast
import subprocess
import sys
import importlib.util
from pathlib import Path


# ============================================================
# 1) PATHS
# ============================================================

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)

MLP43_SCRIPT = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "linear_mlp"
    / "seed_43"
    / "train_linear_mlp_seed43.py"
)

assert MLP43_SCRIPT.is_file(), (
    "Seed-43 MLP trainer not found.\n"
    "Run and finish MLP Seed 43 first.\n\n"
    f"Expected:\n{MLP43_SCRIPT}"
)


# ============================================================
# 2) READ EXACT SEED-43 MLP TRAINER
# ============================================================

original_source = MLP43_SCRIPT.read_text(
    encoding="utf-8"
)

original_tree = ast.parse(
    original_source
)


top_level_classes = {
    node.name: node
    for node in original_tree.body
    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert "Encoder" in top_level_classes
assert "LinearMLP" in top_level_classes

assert "SEED = 43" in original_source

assert '"epochs": 120' in original_source
assert '"patience": 18' in original_source

assert (
    '"pair": "z1,z2,abs(z1-z2),z1*z2"'
    in original_source
)

assert (
    "SCAFFOLD_LINEAR_MLP_SEED_43_COMPLETE.zip"
    in original_source
)


print(
    " Exact MLP Seed-43 trainer loaded.",
    flush=True,
)


# ============================================================
# 3) TORCH GEOMETRIC
# ============================================================

if importlib.util.find_spec(
    "torch_geometric"
) is None:

    print(
        "Installing torch-geometric...",
        flush=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "torch-geometric==2.8.0.post1",
        ],
        check=True,
        timeout=300,
    )


print(
    " torch_geometric available.",
    flush=True,
)


# ============================================================
# 4) GRAPH ENCODER
# ============================================================

graph_encoder = r'''
from torch_geometric.nn import GCNConv


# ============================================================
# TRAIN-ONLY GRAPH CONSTRUCTION
# ============================================================

train_pairs_global_cpu = torch.as_tensor(
    pair_arrays["train"],
    dtype=torch.long,
)

train_pairs_local_cpu = torch.as_tensor(
    train_local,
    dtype=torch.long,
)


# ------------------------------------------------------------
# Global-index graph
# Used during evaluation with x_all
# ------------------------------------------------------------

edge_global = torch.cat(
    [
        train_pairs_global_cpu.T,
        train_pairs_global_cpu.flip(1).T,
    ],
    dim=1,
).contiguous().to(device)


# ------------------------------------------------------------
# Local Train-drug graph
# Used during training with x_train
# ------------------------------------------------------------

edge_train = torch.cat(
    [
        train_pairs_local_cpu.T,
        train_pairs_local_cpu.flip(1).T,
    ],
    dim=1,
).contiguous().to(device)


# ============================================================
# GRAPH QA
# ============================================================

assert edge_global.shape == (
    2,
    2 * len(labels["train"]),
)

assert edge_train.shape == edge_global.shape

assert edge_train.min().item() >= 0

assert edge_train.max().item() < len(train_ids)


train_ids_tensor = torch.as_tensor(
    train_ids,
    dtype=torch.long,
    device=device,
)


# local graph -> exact same global graph
assert torch.equal(
    train_ids_tensor[edge_train],
    edge_global,
)


# ------------------------------------------------------------
# Every interaction edge belongs to a Train drug
# ------------------------------------------------------------

edge_drugs = set(
    edge_global
    .detach()
    .cpu()
    .numpy()
    .ravel()
    .tolist()
)

assert edge_drugs == set(
    train_ids.tolist()
)


# ------------------------------------------------------------
# Critical scaffold leakage check
# ------------------------------------------------------------

test_drugs = set(
    pair_arrays["test"]
    .ravel()
    .tolist()
)

assert not edge_drugs.intersection(
    test_drugs
), (
    "Leakage: TEST drug found in TRAIN interaction graph."
)


print(
    "Directed Train edges:",
    edge_global.shape[1],
    flush=True,
)

print(
    "Train graph drugs:",
    len(edge_drugs),
    flush=True,
)

print(
    "Test drugs incident to Train edges: 0",
    flush=True,
)


# ============================================================
# CONTROLLED GCN ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.layers = nn.ModuleList(
            [
                GCNConv(
                    2250,
                    384,
                    cached=False,
                    add_self_loops=True,
                    normalize=True,
                ),

                GCNConv(
                    384,
                    384,
                    cached=False,
                    add_self_loops=True,
                    normalize=True,
                ),

                GCNConv(
                    384,
                    192,
                    cached=False,
                    add_self_loops=True,
                    normalize=True,
                ),
            ]
        )

        self.norms = nn.ModuleList(
            [
                nn.BatchNorm1d(384),
                nn.BatchNorm1d(384),
                nn.BatchNorm1d(192),
            ]
        )

        self.dropout = nn.Dropout(
            0.35
        )


    def forward(self, x):

        if self.training:

            # Training:
            # only Train-drug feature tensor + Train graph
            assert x.shape[0] == len(train_ids)

            edges = edge_train

        else:

            # Evaluation:
            # all molecular feature rows can be encoded,
            # but only TRAIN interaction edges are present.
            #
            # Held-out drugs therefore have self-loops only.
            assert x.shape[0] == len(all_drugs)

            edges = edge_global


        for i, (layer, norm) in enumerate(
            zip(
                self.layers,
                self.norms,
            )
        ):

            x = layer(
                x,
                edges,
            )

            x = norm(
                x
            )

            if i < 2:

                x = torch.relu(
                    x
                )

                x = self.dropout(
                    x
                )


        return x
'''


# ============================================================
# 5) REPLACE ONLY THE REAL Encoder CLASS
# ============================================================

encoder_node = top_level_classes[
    "Encoder"
]

source_lines = original_source.splitlines(
    keepends=True
)

encoder_start = (
    encoder_node.lineno - 1
)

encoder_end = (
    encoder_node.end_lineno
)


old_encoder = "".join(
    source_lines[
        encoder_start:
        encoder_end
    ]
)


assert old_encoder.lstrip().startswith(
    "class Encoder"
)


graph_source = (
    "".join(
        source_lines[:encoder_start]
    )
    +
    graph_encoder.strip()
    +
    "\n\n"
    +
    "".join(
        source_lines[encoder_end:]
    )
)


# ============================================================
# 6) CHECK AFTER ENCODER REPLACEMENT
# ============================================================

check_tree = ast.parse(
    graph_source
)

check_classes = {
    node.name
    for node in check_tree.body
    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert "Encoder" in check_classes
assert "LinearMLP" in check_classes


print(
    " Only Encoder replaced.",
    flush=True,
)


# ============================================================
# 7) RENAME MODEL + OUTPUT NAMESPACE
# ============================================================

graph_source = graph_source.replace(
    "LinearMLP",
    "ControlledGCN",
)

graph_source = graph_source.replace(
    "linear_mlp",
    "graph",
)

graph_source = graph_source.replace(
    "LINEAR MLP",
    "GRAPH",
)

graph_source = graph_source.replace(
    "SCAFFOLD_LINEAR_MLP_SEED_43_COMPLETE.zip",
    "SCAFFOLD_GRAPH_SEED_43_COMPLETE.zip",
)

graph_source = graph_source.replace(
    " No Graph run or Seed 44 started.",
    " Graph Seed 43 finished. No Seed 44 started.",
)


# ============================================================
# 8) ADD GRAPH-SPECIFIC CONFIG BEFORE IDENTITY CREATION
# ============================================================

cuda_anchor = (
    "assert torch.cuda.is_available()"
)

assert graph_source.count(
    cuda_anchor
) == 1


graph_config = '''
# ============================================================
# GRAPH-SPECIFIC CONFIG
# ============================================================

CONFIG.update({

    "model":
        "graph",

    "graph_edges":
        "Train rows plus reverse; duplicates retained",

    "graph_layer":
        "GCNConv",

    "gcn_normalize":
        True,

    "gcn_add_self_loops":
        True,

    "gcn_cached":
        False,

    "evaluation_edges":
        "Train interaction edges only",

    "heldout_drug_graph_access":
        "self-loop only; no held-out interaction edges",

    "training_encoder_scope":
        "all Train drugs, recomputed per pair minibatch",
})


import torch_geometric

CONFIG[
    "torch_geometric_version"
] = str(
    torch_geometric.__version__
)

'''


graph_source = graph_source.replace(
    cuda_anchor,
    graph_config
    +
    "\n"
    +
    cuda_anchor,
    1,
)


# ============================================================
# 9) PARAMETER COUNT LOGGING
# ============================================================

model_anchor = (
    "model = ControlledGCN().to(device)"
)

assert graph_source.count(
    model_anchor
) == 1


parameter_logging = r'''

# ============================================================
# PARAMETER COUNT
# ============================================================

actual_parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "GRAPH total parameter count:",
    actual_parameter_count,
    flush=True,
)

print(
    "GRAPH trainable parameter count:",
    trainable_parameter_count,
    flush=True,
)

CONFIG[
    "total_parameter_count"
] = int(
    actual_parameter_count
)

CONFIG[
    "trainable_parameter_count"
] = int(
    trainable_parameter_count
)

'''


graph_source = graph_source.replace(
    model_anchor,
    model_anchor
    +
    parameter_logging,
    1,
)


# ============================================================
# 10) FINAL SCRIPT QA
# ============================================================

ast.parse(
    graph_source
)

final_tree = ast.parse(
    graph_source
)

final_classes = {
    node.name
    for node in final_tree.body
    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert "Encoder" in final_classes
assert "ControlledGCN" in final_classes
assert "LinearMLP" not in final_classes

assert "SEED = 43" in graph_source

assert '"model": "graph"' in graph_source

assert '"seed_43"' in graph_source

assert (
    "SCAFFOLD_GRAPH_SEED_43_COMPLETE.zip"
    in graph_source
)

assert '"epochs": 120' in graph_source
assert '"patience": 18' in graph_source

assert "cal_macro83" in graph_source

assert (
    '"pair": "z1,z2,abs(z1-z2),z1*z2"'
    in graph_source
)

assert (
    "Test drugs incident to Train edges: 0"
    in graph_source
)


print(
    " Graph Seed-43 generated script passed QA.",
    flush=True,
)


# ============================================================
# 11) GRAPH OUTPUT DIRECTORY
# ============================================================

GRAPH43_OUT = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "graph"
    / "seed_43"
)

GRAPH43_OUT.mkdir(
    parents=True,
    exist_ok=True,
)


GRAPH43_SCRIPT = (
    GRAPH43_OUT
    / "train_graph_seed43.py"
)


# ============================================================
# 12) SAVE SCRIPT
# ============================================================

if GRAPH43_SCRIPT.exists():

    existing = GRAPH43_SCRIPT.read_text(
        encoding="utf-8"
    )

    assert existing == graph_source, (
        "A DIFFERENT Graph Seed-43 script already exists.\n"
        "Stop rather than mix experiment definitions."
    )

    print(
        " Existing Graph Seed-43 script matches.",
        flush=True,
    )

else:

    GRAPH43_SCRIPT.write_text(
        graph_source,
        encoding="utf-8",
    )

    print(
        " Graph Seed-43 script saved:",
        GRAPH43_SCRIPT,
        flush=True,
    )


# ============================================================
# 13) SAVE EXACT MLP REFERENCE
# ============================================================

REFERENCE_PATH = (
    GRAPH43_OUT
    / "reference_mlp_seed43_training_cell.txt"
)


if REFERENCE_PATH.exists():

    assert (
        REFERENCE_PATH.read_text(
            encoding="utf-8"
        )
        ==
        original_source
    )

else:

    REFERENCE_PATH.write_text(
        original_source,
        encoding="utf-8",
    )


# ============================================================
# 14) PRE-RUN SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80,
    flush=True,
)

print(
    "CORRECTED SCAFFOLD — GRAPH — SEED 43",
    flush=True,
)

print(
    "=" * 80,
    flush=True,
)

print(
    "Max epochs: 120",
    flush=True,
)

print(
    "Early stopping patience: 18",
    flush=True,
)

print(
    " Same Seed-43 initialization protocol",
    flush=True,
)

print(
    " Same corrected scaffold splits",
    flush=True,
)

print(
    " Same train-only molecular preprocessing",
    flush=True,
)

print(
    " Same class weights",
    flush=True,
)

print(
    " Same pair classifier",
    flush=True,
)

print(
    " Same optimizer and scheduler",
    flush=True,
)

print(
    " Only encoder changed: Linear -> GCN",
    flush=True,
)

print(
    " Train interaction edges only",
    flush=True,
)

print(
    " Reverse Train edges included",
    flush=True,
)

print(
    " Duplicate Train edges retained",
    flush=True,
)

print(
    " TEST interaction edges excluded",
    flush=True,
)

print(
    "\nStarting GRAPH — Seed 43...",
    flush=True,
)


# ============================================================
# 15) EXECUTE GRAPH SEED 43
# ============================================================

graph_namespace = {
    "__name__": "__main__",
    "__file__": str(
        GRAPH43_SCRIPT
    ),
}


exec(
    compile(
        graph_source,
        str(GRAPH43_SCRIPT),
        "exec",
    ),
    graph_namespace,
)

Source cell: 63.


In [ ]:
# ============================================================
# CORRECTED NONCHIRAL SCAFFOLD — LINEAR MLP, SEED 44
# Clone exact validated Seed-43 trainer.
# Change ONLY seed/output namespace.
# ============================================================

import ast
from pathlib import Path

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)

SOURCE43 = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "linear_mlp"
    / "seed_43"
    / "train_linear_mlp_seed43.py"
)

assert SOURCE43.is_file(), f"Missing:\n{SOURCE43}"

source43 = SOURCE43.read_text(encoding="utf-8")
ast.parse(source43)

# ---------------- QA source ----------------
assert "SEED = 43" in source43
assert '"seed_43"' in source43
assert "class LinearMLP(nn.Module):" in source43
assert '"epochs": 120' in source43
assert '"patience": 18' in source43
assert "SCAFFOLD_LINEAR_MLP_SEED_43_COMPLETE.zip" in source43

print(" Validated MLP Seed-43 trainer loaded.", flush=True)

# ============================================================
# Build Seed 44
# ============================================================

source44 = source43

assert source44.count("SEED = 43") == 1

source44 = source44.replace(
    "SEED = 43",
    "SEED = 44",
    1,
)

source44 = source44.replace(
    '"seed_43"',
    '"seed_44"',
)

source44 = source44.replace(
    "SCAFFOLD_LINEAR_MLP_SEED_43_COMPLETE.zip",
    "SCAFFOLD_LINEAR_MLP_SEED_44_COMPLETE.zip",
)

source44 = source44.replace(
    "Selection: CAL Macro83 | Seed: 43",
    "Selection: CAL Macro83 | Seed: 44",
)

source44 = source44.replace(
    " LINEAR MLP — SEED 43 COMPLETED",
    " LINEAR MLP — SEED 44 COMPLETED",
)

source44 = source44.replace(
    " No Graph run or Seed 44 started.",
    " No Graph run after Seed 44 started.",
)

# ============================================================
# Final QA
# ============================================================

ast.parse(source44)

assert "SEED = 44" in source44
assert "SEED = 43" not in source44

assert '"seed_44"' in source44

assert "class LinearMLP(nn.Module):" in source44
assert '"epochs": 120' in source44
assert '"patience": 18' in source44

assert (
    '"pair": "z1,z2,abs(z1-z2),z1*z2"'
    in source44
)

assert (
    "SCAFFOLD_LINEAR_MLP_SEED_44_COMPLETE.zip"
    in source44
)

# ============================================================
# Save
# ============================================================

OUT44 = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "linear_mlp"
    / "seed_44"
)

OUT44.mkdir(parents=True, exist_ok=True)

SCRIPT44 = (
    OUT44
    / "train_linear_mlp_seed44.py"
)

if SCRIPT44.exists():
    existing = SCRIPT44.read_text(encoding="utf-8")
    assert existing == source44, (
        "Different Seed-44 MLP script already exists."
    )
else:
    SCRIPT44.write_text(
        source44,
        encoding="utf-8"
    )

# preserve parent script
REFERENCE = (
    OUT44
    / "reference_mlp_seed43_training_cell.txt"
)

if REFERENCE.exists():
    assert REFERENCE.read_text(encoding="utf-8") == source43
else:
    REFERENCE.write_text(source43, encoding="utf-8")

print("=" * 80)
print("CORRECTED SCAFFOLD — LINEAR MLP — SEED 44")
print("=" * 80)
print(" Same split")
print(" Same features")
print(" Same architecture")
print(" Same class weights")
print(" Same optimizer/scheduler")
print(" 120 max epochs")
print(" Patience = 18")
print(" Only random seed/output changed")
print("\nStarting LINEAR MLP — Seed 44...", flush=True)

namespace44 = {
    "__name__": "__main__",
    "__file__": str(SCRIPT44),
}

exec(
    compile(
        source44,
        str(SCRIPT44),
        "exec",
    ),
    namespace44,
)

Source cell: 64.


In [ ]:
# ============================================================
# CORRECTED NONCHIRAL SCAFFOLD — GRAPH, SEED 44
# Clone exact validated Graph Seed-43 trainer.
# Change ONLY seed/output namespace.
# ============================================================

import ast
from pathlib import Path

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)

SOURCE43 = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "graph"
    / "seed_43"
    / "train_graph_seed43.py"
)

assert SOURCE43.is_file(), f"Missing:\n{SOURCE43}"

source43 = SOURCE43.read_text(
    encoding="utf-8"
)

ast.parse(source43)

# ============================================================
# Verify exact successful Graph-43 protocol
# ============================================================

assert "SEED = 43" in source43
assert '"seed_43"' in source43

assert "class ControlledGCN(nn.Module):" in source43
assert "class LinearMLP" not in source43

assert "GCNConv" in source43

assert (
    "Test drugs incident to Train edges: 0"
    in source43
)

assert '"epochs": 120' in source43
assert '"patience": 18' in source43

assert (
    "SCAFFOLD_GRAPH_SEED_43_COMPLETE.zip"
    in source43
)

print(
    " Validated Graph Seed-43 trainer loaded.",
    flush=True,
)

# ============================================================
# Build Graph Seed 44
# ============================================================

source44 = source43

assert source44.count(
    "SEED = 43"
) == 1

source44 = source44.replace(
    "SEED = 43",
    "SEED = 44",
    1,
)

source44 = source44.replace(
    '"seed_43"',
    '"seed_44"',
)

source44 = source44.replace(
    "SCAFFOLD_GRAPH_SEED_43_COMPLETE.zip",
    "SCAFFOLD_GRAPH_SEED_44_COMPLETE.zip",
)

source44 = source44.replace(
    "Selection: CAL Macro83 | Seed: 43",
    "Selection: CAL Macro83 | Seed: 44",
)

source44 = source44.replace(
    " GRAPH — SEED 43 COMPLETED",
    " GRAPH — SEED 44 COMPLETED",
)

source44 = source44.replace(
    " Graph Seed 43 finished. No Seed 44 started.",
    " Graph Seed 44 finished. All scaffold seeds completed.",
)

# ============================================================
# Final QA
# ============================================================

ast.parse(source44)

assert "SEED = 44" in source44
assert "SEED = 43" not in source44

assert '"seed_44"' in source44

assert "class ControlledGCN(nn.Module):" in source44
assert "class LinearMLP" not in source44

assert "GCNConv" in source44

assert (
    "Test drugs incident to Train edges: 0"
    in source44
)

assert '"epochs": 120' in source44
assert '"patience": 18' in source44

assert (
    '"pair": "z1,z2,abs(z1-z2),z1*z2"'
    in source44
)

assert (
    "SCAFFOLD_GRAPH_SEED_44_COMPLETE.zip"
    in source44
)

# ============================================================
# Save Graph Seed 44
# ============================================================

OUT44 = (
    PROTOCOL
    / "controlled_rerun_v1"
    / "graph"
    / "seed_44"
)

OUT44.mkdir(
    parents=True,
    exist_ok=True,
)

SCRIPT44 = (
    OUT44
    / "train_graph_seed44.py"
)

if SCRIPT44.exists():

    existing = SCRIPT44.read_text(
        encoding="utf-8"
    )

    assert existing == source44, (
        "Different Graph Seed-44 script already exists."
    )

else:

    SCRIPT44.write_text(
        source44,
        encoding="utf-8"
    )

# Preserve exact Graph-43 parent
REFERENCE = (
    OUT44
    / "reference_graph_seed43_training_cell.txt"
)

if REFERENCE.exists():

    assert (
        REFERENCE.read_text(
            encoding="utf-8"
        )
        ==
        source43
    )

else:

    REFERENCE.write_text(
        source43,
        encoding="utf-8"
    )

# ============================================================
# Start Graph 44
# ============================================================

print("=" * 80)
print("CORRECTED SCAFFOLD — GRAPH — SEED 44")
print("=" * 80)

print(" Same corrected scaffold split")
print(" Same train-only preprocessing")
print(" Same molecular features")
print(" Same GCN architecture")
print(" Same Train-only interaction graph")
print(" Same pair classifier")
print(" Same class weights")
print(" Same optimizer/scheduler")
print(" 120 max epochs")
print(" Patience = 18")
print(" Only random seed/output changed")

print(
    "\nStarting GRAPH — Seed 44...",
    flush=True,
)

namespace44 = {
    "__name__": "__main__",
    "__file__": str(SCRIPT44),
}

exec(
    compile(
        source44,
        str(SCRIPT44),
        "exec",
    ),
    namespace44,
)